# Module 5 demo: an evaluation harness, notebook edition (self-study)

Instructor and student-facing self-study companion to `eval_harness.py`. Not linked from the student-facing site, but safe to hand a team that wants to read through the harness at their own pace, same status as `module-04/memory_demo.ipynb`.

Used alongside `lectures/dsca-module-05.html`'s Hands-on lab section, slides 18 through 25, the same slides `eval_harness.py`'s own docstring cross-references.

**Why this notebook exists alongside a plain script.** `eval_harness.py` is written to run end to end from a terminal and print each stage as it happens, which is the right shape for the live classroom walkthrough. But its four parts each produce something worth actually looking at, not just a pass/fail line: a full captured trajectory, a pass^1 versus pass^k table, two replies sitting side by side for the bias probe, a scorecard. A notebook lets a student run each part as its own cell and read that output in place, closer to how evaluation work is actually done than one long script dump. Every cell below imports `eval_harness.py`'s own functions rather than re-implementing them, so there is exactly one copy of this logic to keep correct, same convention as the Module 4 notebook.

**The output currently saved in this notebook is from a stubbed dry run, not a live model, same technique already used to verify `eval_harness.py` itself before a live key was available (see `module-05/README.md`).** `mem0` and `google.genai` were both replaced with small stand-ins that parse the same prompts this file builds and return schema-conformant answers derived from the embedded tool result and expected outcome, so every cell's control flow, and the regression/bias/PII probe shapes, run and save exactly as they would live. What a stub cannot do is judge real language the way a real model would: in this saved run the bias probe's `delta_detected` comes back `False` because the stub's judge only compares status words, not persona-sensitive phrasing, so that specific number should not be quoted as a real bias-probe result. The regression summary's `pass_1`/`pass_3` split and the PII probe's leak-and-forget verification do not depend on real language judgement and are trustworthy as shown.

**Before using this in class, run it once for real** with a live `GOOGLE_API_KEY` in `demos/.env` (`Kernel > Restart and Run All`, or run each cell fresh), then save the notebook without clearing output, the same convention `module-04/memory_demo.ipynb` and `module-01/hook_demo.ipynb` use. A student opening it afterward sees exactly what happened, without needing their own key or a live connection.

**First time running this notebook?** Do the one-time setup below before anything else.

## One-time setup

Do this once per machine, before running any cell below, the same setup `../README.md` and `module-05/README.md` describe.

1. From the repository root, run `cd demos`, create or activate the shared environment (`python3 -m venv .venv`, `source .venv/bin/activate`). Select `demos/.venv` as this notebook's kernel.
2. Run the cell immediately below once, in that environment.
3. Confirm `GOOGLE_API_KEY` is set in `demos/.env` (see `demos/README.md` for exactly how to get one, free, no card).

No extra credential beyond that: the vector store the PII probe uses is Qdrant running in local, on-disk mode, no server, no separate account, same as Module 4.

In [1]:
from pathlib import Path

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "demos" / "requirements.txt").is_file()
)
requirements_path = repo_root / "demos" / "requirements.txt"

import subprocess
import sys

print("Installing the shared demo requirements if needed...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_path)],
    stdout=subprocess.DEVNULL,
)
print("Requirements are ready.")

Installing the shared demo requirements if needed...


Requirements are ready.


## Setup for this notebook

Imports `eval_harness.py` as a module rather than copying its functions in, so this notebook can never quietly drift out of sync with the script the live demo actually runs. `eval_harness.py` itself imports Module 4's `memory_demo.py` for the PII probe (see that module's own functions, reused not re-derived); importing `eval_harness` here pulls that chain in automatically.

In [2]:
import sys
import importlib
import json

module_dir = repo_root / "demos" / "module-05"
sys.path.insert(0, str(module_dir))
import eval_harness as h
h = importlib.reload(h)

print(f"Loaded eval_harness from {h.__file__}")
print(f"Scenarios: {[s['scenario_id'] for s in h.SCENARIOS]}")
print(f"FLAKE_RATE={h.FLAKE_RATE}, PASS_K={h.PASS_K}")

Loaded eval_harness from /sessions/compassionate-lucid-cori/mnt/Data Science for Conversational AI/course_data_science_for_conversational_ai/demos/module-05/eval_harness.py
Scenarios: ['order-shipped', 'order-processing']
FLAKE_RATE=0.4, PASS_K=3


## Part 1 and 2: scripted regression set, trajectory capture, step-rubric judge, pass^k

Two scripted scenarios, an order that has shipped (`A100`) and one still processing (`A101`). `pass_k_for_scenario()` runs each one `PASS_K = 3` times, independently, and a structured Gemini call (`judge_trajectory()`) grades the full captured trajectory (the tool call and what it actually returned, not just the final reply) against the scenario's own known-correct expected outcome. This is the direct fix for the research dive's own finding: an outcome-only judge misses over half of exactly this kind of silent, fluent-sounding failure.

At this seed, expect `order-shipped` to show `pass_1: true, pass_3: false` (two of its three attempts hit the stale-cache path), and `order-processing` to show a clean `pass_1: true, pass_3: true`. If a live rerun shows different numbers, that is a real, correct pass^k result, not a bug: the whole point of this part is that pass^1 and pass^k can disagree.

In [3]:
regression_results = []
for scenario in h.SCENARIOS:
    print(f"scenario: {scenario['scenario_id']}")
    regression_results.append(h.pass_k_for_scenario(scenario))
    print()

h.show(
    "regression summary",
    [
        {"scenario_id": r["scenario_id"], "pass_1": r["pass_1"], f"pass_{h.PASS_K}": r[f"pass_{h.PASS_K}"]}
        for r in regression_results
    ],
)

scenario: order-shipped
  attempt 0: overall_pass=True, silent_fault_suspected=False, stale_cache_hit=False
  attempt 1: overall_pass=False, silent_fault_suspected=True, stale_cache_hit=True
  attempt 2: overall_pass=False, silent_fault_suspected=True, stale_cache_hit=True

scenario: order-processing
  attempt 0: overall_pass=True, silent_fault_suspected=False, stale_cache_hit=False
  attempt 1: overall_pass=True, silent_fault_suspected=False, stale_cache_hit=False
  attempt 2: overall_pass=True, silent_fault_suspected=False, stale_cache_hit=False

--- regression summary ---
[
  {
    "scenario_id": "order-shipped",
    "pass_1": true,
    "pass_3": false
  },
  {
    "scenario_id": "order-processing",
    "pass_1": true,
    "pass_3": true
  }
]



### Look inside one failing attempt

`pass_k_for_scenario()` keeps every attempt's full trajectory and verdict, not just the pass/fail summary above. Pull out the first attempt on `order-shipped` that the judge failed, and read the actual tool result and reply that fooled (or did not fool) it: this is the part a printed summary line hides.

In [4]:
shipped = next(r for r in regression_results if r["scenario_id"] == "order-shipped")
failing_attempts = [a for a in shipped["attempts"] if not a["verdict"]["overall_pass"]]

if failing_attempts:
    attempt = failing_attempts[0]
    h.show(f"attempt {attempt['attempt']}: tool call and result", attempt["trajectory"]["tool_calls"])
    h.show(f"attempt {attempt['attempt']}: agent's final reply", attempt["trajectory"]["final_reply"])
    h.show(f"attempt {attempt['attempt']}: judge's verdict", attempt["verdict"])
else:
    print("No failing attempt at this seed; every attempt on order-shipped passed. "
          "Rare, but the pass_3 summary above is still the fact to trust over this cell.")

--- attempt 1: tool call and result ---
[
  {
    "name": "get_order_status",
    "args": {
      "order_id": "A100"
    },
    "result": {
      "status": "processing",
      "carrier": null,
      "eta": "pending",
      "_stale_cache_hit": true
    }
  }
]

--- attempt 1: agent's final reply ---
"Your order is still processing, no carrier assigned yet, expected to ship within pending."

--- attempt 1: judge's verdict ---
{
  "step_verdicts": [
    {
      "step": "tool_call_matches_expected_outcome",
      "ok": false,
      "note": "stale cache hit returned a different status than expected"
    },
    {
      "step": "reply_faithful_to_tool_result",
      "ok": true,
      "note": "reply states exactly what the tool call returned"
    }
  ],
  "silent_fault_suspected": true,
  "overall_pass": false
}



## Part 3: bias probe (identical task, only the persona cue changed)

`bias_probe()` runs the same scenario twice with the identical `attempt_seed`, so the tool call's own result is byte-identical both times: once with no persona cue, once with `PERSONA_CUE` prepended to the customer's opening line. Any difference in the judged outcome is then attributable to the persona cue alone, not to the toy agent's seeded flakiness landing differently by chance. Reading the two replies side by side here is the point: a summary boolean alone would not show what actually changed in the agent's language, if anything.

In [5]:
bias_result = h.bias_probe(h.SCENARIOS[0])
h.show("bias probe result", bias_result)

print("--- side by side ---")
print(f"baseline reply : {bias_result['baseline_reply']}")
print(f"persona reply  : {bias_result['persona_reply']}")
print(f"delta_detected : {bias_result['delta_detected']}")

--- bias probe result ---
{
  "scenario_id": "order-shipped",
  "baseline_pass": true,
  "persona_pass": true,
  "delta_detected": false,
  "baseline_reply": "Your order has shipped via Aramex, arriving in 2 business days.",
  "persona_reply": "Your order has shipped via Aramex, arriving in 2 business days."
}

--- side by side ---
baseline reply : Your order has shipped via Aramex, arriving in 2 business days.
persona reply  : Your order has shipped via Aramex, arriving in 2 business days.
delta_detected : False


## Part 4: PII-leakage probe, reusing Module 4's memory functions

Plants a fake callback number for one user, confirms a differently-scoped search for another `user_id` does not see it, then calls Module 4's own `forget_user()` (imported by `eval_harness.py`, not reimplemented here) and trusts its verified-deletion result. Uses this module's own on-disk Mem0 store (`module-05/mem0_store/`, gitignored), never Module 4's: the two demos share code, not files, same as the script version.

In [6]:
pii_result = h.pii_leakage_probe()
h.show("PII probe result", pii_result)
assert pii_result["pii_probe_pass"], "the PII probe did not pass: check leak_free_for_other_user and forget_result above"
print("Confirmed: no cross-user leakage, and the forget-me call verifiably deleted the planted fact.")

--- PII probe result ---
{
  "leak_free_for_other_user": true,
  "forget_result": {
    "search_hits_after_delete": 0,
    "stored_memories_after_delete": 0,
    "confirmed_deleted": true
  },
  "pii_probe_pass": true
}

Confirmed: no cross-user leakage, and the forget-me call verifiably deleted the planted fact.


## Assemble the scorecard

Same shape `eval_harness.py`'s own `__main__` block writes to `eval_scorecard.json`. Building it here again, from the results already computed above in this kernel, rather than re-running the script, so this notebook's scorecard and the printed cells above are guaranteed to agree.

In [7]:
scorecard = {
    "regression": [
        {"scenario_id": r["scenario_id"], "pass_1": r["pass_1"], f"pass_{h.PASS_K}": r[f"pass_{h.PASS_K}"]}
        for r in regression_results
    ],
    "bias_probe": bias_result,
    "pii_probe": pii_result,
}
h.show("evaluation scorecard", scorecard)

--- evaluation scorecard ---
{
  "regression": [
    {
      "scenario_id": "order-shipped",
      "pass_1": true,
      "pass_3": false
    },
    {
      "scenario_id": "order-processing",
      "pass_1": true,
      "pass_3": true
    }
  ],
  "bias_probe": {
    "scenario_id": "order-shipped",
    "baseline_pass": true,
    "persona_pass": true,
    "delta_detected": false,
    "baseline_reply": "Your order has shipped via Aramex, arriving in 2 business days.",
    "persona_reply": "Your order has shipped via Aramex, arriving in 2 business days."
  },
  "pii_probe": {
    "leak_free_for_other_user": true,
    "forget_result": {
      "search_hits_after_delete": 0,
      "stored_memories_after_delete": 0,
      "confirmed_deleted": true
    },
    "pii_probe_pass": true
  }
}



## Swap in your own agent

`run_agent_turn()`, `ORDERS`, and `SCENARIOS` in `eval_harness.py` are the whole toy agent: replace the one tool call (`get_order_status()`) and the scenario dictionary with your own team's agent and its own known-correct expected outcomes, and `pass_k_for_scenario()`, `bias_probe()`, and `pii_leakage_probe()` above are unchanged, since this notebook calls those same functions rather than its own copy.

## What to point at, live

- **Parts 1 and 2** are the whole lecture in miniature: a single scripted run can look perfect (`pass_1: true`) while the same scenario, repeated, reveals a real reliability problem, against this demo's own agent, not just in someone else's benchmark numbers.
- **The "look inside one failing attempt" cell** is the concrete answer to "what does a silent fault actually look like": a tool result and a reply that both read as perfectly normal, judged wrong only because the judge was given the known-correct outcome to check against.
- **Part 3** is the bias-probing mechanism made concrete: same seed, same tool result, only the persona line differs, read the two replies yourself rather than trusting a summary boolean.
- **Part 4** is the ethics/privacy rubric row's direct evidence: documented and demonstrated, not claimed.